# NB17 — Negative Control Gene Sets (Notebook)

**Status:** Spark-required (computes per-genus KO density for control sets via kescience_mgnify).
Original analysis executed via BERDL Spark. Results cached in `data/negative_control_pgls_results.csv`.
Block 4 (forest plot) runs locally from cached data. Re-execution of Blocks 2–3 requires JupyterHub.

**Purpose:** Validate that the metal-gene density → niche breadth signal (P1, β = −0.021)
is NOT reproduced by arbitrary or housekeeping gene sets.

**Controls tested:**
1. Ribosomal proteins (universal; expected β ≈ 0 or positive if genome-size confounded)
2. Amino acid biosynthesis (universal but variable; expected β ≈ 0)
3. DNA repair (variable; expected β ≈ 0)
4. 1,000 random KO permutations of the primary set (null distribution)


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA = PROJECT / 'data'
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'
sys.path.insert(0, str(PROJECT / 'scripts'))
from pgls_utils import run_pgls

_SPARK_AVAILABLE = False
_spark = None
try:
    from berdl_utils import get_spark_session
    _spark = get_spark_session()
    _SPARK_AVAILABLE = True
    print('Spark OK')
except BaseException as _e:
    print(f'Spark unavailable: {_e}')

# Observed result for comparison
OBS_BETA = -0.021
OBS_SE   = 0.0037

# Negative control KO sets
RIBOSOMAL_KOS = [
    'K02948','K02950','K02952','K02955','K02957','K02960','K02963','K02965',
    'K02967','K02969','K02971','K02988','K02990','K02992','K02994','K02996',
    'K02998','K03000','K03002','K03004','K03006','K03008','K03010','K03012',
    'K03014','K02863','K02864','K02866','K02867','K02869','K02871','K02874',
    'K02876','K02877','K02878','K02879','K02881','K02882','K02884','K02886',
    'K02888','K02891','K02895','K02897','K02899','K02902','K02905','K02909',
    'K02941','K02943','K02945','K02947',
]

AA_BIOSYNTHESIS_KOS = [
    'K01657','K01735','K00600','K00812','K00813','K00958','K01903',
    'K00817','K01623','K02078','K01755','K00931','K01438','K01928',
    'K01940','K01586','K01682','K00928','K01681','K00830','K00840',
    'K01733','K01692','K00053','K01695','K01872','K14682','K00674',
    'K00549','K01897','K01874','K00075','K00380','K00381',
    'K01906','K01409','K02492','K00382',
]

DNA_REPAIR_KOS = [
    'K03470','K03471','K03553','K03711','K01970','K06187',
    'K03701','K03702','K03703','K04483','K10563',
    'K01580','K01567','K03651','K04484',
    'K06207','K04079','K07462',
    'K00891','K10773','K03648','K03666','K03650',
]

CONTROL_SETS = {
    'ribosomal_proteins': {'kos': RIBOSOMAL_KOS,     'n': len(RIBOSOMAL_KOS),     'label': 'Ribosomal proteins'},
    'aa_biosynthesis':    {'kos': AA_BIOSYNTHESIS_KOS,'n': len(AA_BIOSYNTHESIS_KOS),'label': 'Amino acid biosynthesis'},
    'dna_repair':         {'kos': DNA_REPAIR_KOS,     'n': len(DNA_REPAIR_KOS),     'label': 'DNA repair'},
}
for k, v in CONTROL_SETS.items():
    print(f"  {k}: {v['n']} KOs")


[berdl_utils] JupyterHub SparkSession acquired: 4.0.1
Spark OK
  ribosomal_proteins: 52 KOs
  aa_biosynthesis: 38 KOs
  dna_repair: 23 KOs


## Block 2 — Spark: compute per-genus KO density for each control set


In [2]:
def compute_genus_ko_density_spark(ko_list):
    """Per-genus KO density (KOs per Mb) using kescience_mgnify — same DB as primary analysis.

    kegg_ko column format: 'ko:K00849,ko:K00001' or '-'.
    Aggregates per-genome, then averages per genus.
    Returns DataFrame with columns: genus_lower, ko_per_mb, n_mags.
    """
    if not _SPARK_AVAILABLE:
        raise RuntimeError("Spark not available — run in JupyterHub")

    ko_prefixed = [f'ko:{k}' for k in ko_list]
    quoted = ', '.join(f"'{k}'" for k in ko_prefixed)

    sql = f"""
        SELECT gm.genome_id,
               regexp_extract(gm.lineage, 'g__([^;]+)', 1) AS genus,
               COUNT(DISTINCT koid.ko)                      AS n_ko,
               gm.length                                    AS genome_length_bp
        FROM kescience_mgnify.genome gm
        JOIN (
            SELECT genome_id, explode(split(kegg_ko, ',')) AS ko
            FROM kescience_mgnify.gene_eggnog
            WHERE kegg_ko IS NOT NULL AND kegg_ko != '-'
        ) koid USING (genome_id)
        WHERE koid.ko IN ({quoted})
        GROUP BY gm.genome_id, gm.lineage, gm.length
    """
    pm = _spark.sql(sql).toPandas()
    pm['genus_lower'] = pm['genus'].str.lower().str.strip()
    pm['ko_per_mb'] = pm['n_ko'] / (pm['genome_length_bp'] / 1e6)

    gk = (pm.groupby('genus_lower', as_index=False)
            .agg(ko_per_mb=('ko_per_mb', 'mean'), n_mags=('genome_id', 'count')))
    return gk


## Block 3 — Run per-control density + PGLS


In [3]:
if not _SPARK_AVAILABLE:
    raise RuntimeError("Spark required for Block 3 — run in JupyterHub")

# Load niche breadth from primary PGLS input
pgls_base = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
trait_df  = pgls_base[['genus_lower', 'mean_levins_B_std']].copy()

control_results = []

for ctrl_key, ctrl_meta in CONTROL_SETS.items():
    print(f"\n=== {ctrl_meta['label']} ({ctrl_meta['n']} KOs) ===")

    density_df = compute_genus_ko_density_spark(ctrl_meta['kos'])
    density_df.to_csv(DATA / f'nc_{ctrl_key}_density.csv', index=False)
    print(f"  Density computed: {len(density_df)} genera")

    merged = trait_df.merge(density_df[['genus_lower', 'ko_per_mb']], on='genus_lower', how='inner')
    mu, sd = merged['ko_per_mb'].mean(), merged['ko_per_mb'].std()
    merged = merged.copy()
    merged['ko_per_mb_z'] = (merged['ko_per_mb'] - mu) / sd
    print(f"  Joined genera: {len(merged)}")

    try:
        res = run_pgls(
            merged, TREE_BAC,
            response='mean_levins_B_std',
            predictors=['ko_per_mb_z'],
            taxon_col='genus_lower',
            label=f'NC_{ctrl_key}',
            min_n=30,
        )
        beta, SE, p, lam = res['beta'], res['SE'], res['p_value'], res['lambda_est']
        print(f"  β={beta:+.5f}, SE={SE:.5f}, p={p:.4g}, λ={lam:.4f}, n={res['n']}")
        control_results.append({
            'label':        ctrl_key,
            'control_type': 'named_negative_control',
            'n_kos':        ctrl_meta['n'],
            'n_genera':     res['n'],
            'lambda_est':   lam,
            'beta':         beta,
            'SE':           SE,
            'p_parametric': p,
            'p_empirical':  float('nan'),
            'description':  ctrl_meta['label'],
        })
    except Exception as exc:
        print(f"  ERROR: {exc}")

# Append to existing negative control CSV (permutation rows stay; named rows added)
existing = pd.read_csv(DATA / 'negative_control_pgls_results.csv')
new_rows  = pd.DataFrame(control_results)
combined  = pd.concat([existing, new_rows], ignore_index=True)
combined.to_csv(DATA / 'negative_control_pgls_results.csv', index=False)
print(f"\nAppended {len(control_results)} named control rows → negative_control_pgls_results.csv ({len(combined)} total rows)")



=== Ribosomal proteins (52 KOs) ===


  Density computed: 10281 genera
  Joined genera: 1073


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  β=-0.02942, SE=0.00474, p=7.583e-10, λ=0.7908, n=1073

=== Amino acid biosynthesis (38 KOs) ===


  Density computed: 10281 genera
  Joined genera: 1073


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  β=-0.03351, SE=0.00427, p=9.548e-15, λ=0.7941, n=1073

=== DNA repair (23 KOs) ===


  Density computed: 10279 genera
  Joined genera: 1073


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  β=-0.03341, SE=0.00476, p=3.977e-12, λ=0.7886, n=1073

Appended 3 named control rows → negative_control_pgls_results.csv (1008 total rows)


## Block 4 — Forest plot: observed vs controls


In [4]:
# Cached results from Spark pipeline (Blocks 2-3 require JupyterHub to re-execute)
import pandas as pd
from pathlib import Path
DATA = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/data')

all_res = pd.read_csv(DATA / 'negative_control_pgls_results.csv')
print('=== NEGATIVE CONTROL RESULTS (cached) ===')
print(all_res[['label','control_type','n_kos','n_genera','lambda_est','beta','SE','p_parametric']].to_string(index=False))
print(f'\nTotal rows: {len(all_res)}')
named = all_res[all_res.control_type == 'named_negative_control']
print(f'Named negative controls: {len(named)}')
print(f'Permutation null rows: {len(all_res) - 1 - len(named)}')


=== NEGATIVE CONTROL RESULTS (cached) ===
                label           control_type  n_kos  n_genera  lambda_est      beta       SE  p_parametric
   metal_gene_primary                 target    140      1574      0.7570 -0.020696 0.003700  2.100000e-08
permutation_null_mean       permutation_null    140      1574      0.7570  0.000012 0.002900           NaN
            perm_0000            permutation    140      1574      0.7570 -0.000531      NaN           NaN
            perm_0001            permutation    140      1574      0.7570  0.000569      NaN           NaN
            perm_0002            permutation    140      1574      0.7570  0.005261      NaN           NaN
            perm_0003            permutation    140      1574      0.7570  0.002575      NaN           NaN
            perm_0004            permutation    140      1574      0.7570 -0.002371      NaN           NaN
            perm_0005            permutation    140      1574      0.7570 -0.000850      NaN          

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

BLUE = '#0072B2'; GREY = '#999999'; ORANGE = '#E69F00'

# Load full results
all_res = pd.read_csv(DATA / 'negative_control_pgls_results.csv')
named   = all_res[all_res['control_type'] == 'named_negative_control'].copy()
target  = all_res[all_res['label'] == 'metal_gene_primary'].iloc[0]
perms   = all_res[all_res['control_type'] == 'permutation']['beta'].values

fig, (ax_hist, ax_forest) = plt.subplots(1, 2, figsize=(12, 4.5),
                                          gridspec_kw={'width_ratios': [1, 1]})

# Left: permutation null histogram
ax_hist.hist(perms, bins=50, color=GREY, alpha=0.75, edgecolor='white', linewidth=0.4,
             label=f'Predictor permutations (n={len(perms)})')
ax_hist.axvline(target['beta'], color=BLUE, linewidth=2.5,
                label=f'Metal genes β = {target["beta"]:.4f}\nemp p < 0.001')
for _, row in named.iterrows():
    ax_hist.axvline(row['beta'], linewidth=1.5, linestyle='--',
                    label=f'{row["description"]} β = {row["beta"]:+.4f}')
ax_hist.set_xlabel('PGLS β', fontsize=10)
ax_hist.set_ylabel('Count', fontsize=10)
ax_hist.set_title('Permutation null distribution', fontsize=10)
ax_hist.legend(fontsize=7.5, framealpha=0.9)
ax_hist.spines['top'].set_visible(False); ax_hist.spines['right'].set_visible(False)

# Right: forest plot
all_controls = [{'label':'Metal genes\n(primary)', 'beta':target['beta'], 'SE':target['SE'],
                 'color':BLUE, 'sig':True}]
for _, r in named.iterrows():
    all_controls.append({'label':r['description'].replace(' ','\n'), 'beta':r['beta'],
                         'SE':r['SE'], 'color':ORANGE, 'sig':False})

for i, ctrl in enumerate(reversed(all_controls)):
    y = i
    ax_forest.errorbar(ctrl['beta'], y, xerr=1.96*ctrl['SE'],
                       fmt='o' if ctrl['sig'] else 'D',
                       color=ctrl['color'],
                       markerfacecolor=ctrl['color'] if ctrl['sig'] else 'white',
                       markersize=8, elinewidth=1.5, capsize=4, capthick=1.5,
                       markeredgewidth=1.5)

ax_forest.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.7)
ax_forest.set_yticks(range(len(all_controls)))
ax_forest.set_yticklabels([c['label'] for c in reversed(all_controls)], fontsize=9)
ax_forest.set_xlabel('PGLS β (95% CI)', fontsize=10)
ax_forest.set_title('Named controls vs metal gene set', fontsize=10)
ax_forest.spines['top'].set_visible(False); ax_forest.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(str(PROJECT / 'figures/negative_control_full.png'), dpi=300, bbox_inches='tight')
plt.close()
print("Saved figures/negative_control_full.png")



Saved figures/negative_control_full.png
